In [1]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
import time
import os
import sys

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "14g") \
    .config("spark.executor.memory", "14g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/04/07 23:24:15 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/04/07 23:24:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/07 23:24:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [4]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
# Get unique labels and their counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the results
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
# List of labels to drop
labels_to_drop = ["Defense Evasion", "Exfiltration", "Initial Access", "Lateral Movement", "Persistence", "Privilege Escalation", "Resource Development", "Credential Access"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
filtered_label_counts.show()

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|     Discovery|   2086|
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+



In [7]:
# Drop the datetime column
#df = df.drop("datetime")
df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
df_indexed.show()

24/04/07 23:25:52 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB


+---------+-------------+----------+------------+--------------------+---------+-------------+--------------+----------+----------+----------+-------------+-------------------+---------------+------------------+---------------+-------------+--------------------+--------------------+-----------+-------------------+--------------------+----------------+
|resp_pkts|orig_ip_bytes|local_resp|missed_bytes|            duration|orig_pkts|resp_ip_bytes|dest_port_zeek|orig_bytes|local_orig|resp_bytes|src_port_zeek|                 ts|service_indexed|conn_state_indexed|history_indexed|proto_indexed|dest_ip_zeek_indexed|community_id_indexed|uid_indexed|src_ip_zeek_indexed|label_tactic_indexed|datetime_indexed|
+---------+-------------+----------+------------+--------------------+---------+-------------+--------------+----------+----------+----------+-------------+-------------------+---------------+------------------+---------------+-------------+--------------------+--------------------+---------

In [8]:
# Split the data into training and test sets
train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

In [9]:
from pyspark.ml.feature import Imputer

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)

# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
train_data_imputed.show()
test_data_imputed.show()

24/04/07 23:26:12 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:27:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/04/07 23:27:29 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


+---------+-------------+----------+------------+--------+---------+-------------+--------------+----------+----------+----------+-------------+-------------------+---------------+------------------+---------------+-------------+--------------------+--------------------+-----------+-------------------+--------------------+----------------+-----------------+---------------------+--------------------+------------------+-----------------+---------------------+----------------------+------------------+------------------+---------------------+-------------------+
|resp_pkts|orig_ip_bytes|local_resp|missed_bytes|duration|orig_pkts|resp_ip_bytes|dest_port_zeek|orig_bytes|local_orig|resp_bytes|src_port_zeek|                 ts|service_indexed|conn_state_indexed|history_indexed|proto_indexed|dest_ip_zeek_indexed|community_id_indexed|uid_indexed|src_ip_zeek_indexed|label_tactic_indexed|datetime_indexed|resp_pkts_imputed|orig_ip_bytes_imputed|missed_bytes_imputed|  duration_imputed|orig_pkts_impu

24/04/07 23:28:03 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


+---------+-------------+----------+------------+--------+---------+-------------+--------------+----------+----------+----------+-------------+-------------------+---------------+------------------+---------------+-------------+--------------------+--------------------+-----------+-------------------+--------------------+----------------+-----------------+---------------------+--------------------+------------------+-----------------+---------------------+----------------------+------------------+------------------+---------------------+-------------------+
|resp_pkts|orig_ip_bytes|local_resp|missed_bytes|duration|orig_pkts|resp_ip_bytes|dest_port_zeek|orig_bytes|local_orig|resp_bytes|src_port_zeek|                 ts|service_indexed|conn_state_indexed|history_indexed|proto_indexed|dest_ip_zeek_indexed|community_id_indexed|uid_indexed|src_ip_zeek_indexed|label_tactic_indexed|datetime_indexed|resp_pkts_imputed|orig_ip_bytes_imputed|missed_bytes_imputed|  duration_imputed|orig_pkts_impu

In [10]:
from pyspark.ml.feature import VectorAssembler

# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)

# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
test_data_assembled.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = false)

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = false)



In [11]:
"""
# Get the distinct values of the 'label_tactic_indexed' column
distinct_classes = train_data_assembled.select("label_tactic_indexed").distinct().collect()

# Convert the result to a list of unique classes
unique_classes = [row['label_tactic_indexed'] for row in distinct_classes]

# Print the unique classes
print("Unique classes:", unique_classes)

# Group by the 'label_tactic_indexed' column and count the occurrences of each class
class_counts = train_data_assembled.groupBy("label_tactic_indexed").count()

# Show the class counts
class_counts.show()

# Get the distinct values of the 'label_tactic_indexed' column
distinct_classes = test_data_assembled.select("label_tactic_indexed").distinct().collect()

# Convert the result to a list of unique classes
unique_classes = [row['label_tactic_indexed'] for row in distinct_classes]

# Print the unique classes
print("Unique classes:", unique_classes)

# Group by the 'label_tactic_indexed' column and count the occurrences of each class
class_counts = test_data_assembled.groupBy("label_tactic_indexed").count()

# Show the class counts
class_counts.show()
"""

'\n# Get the distinct values of the \'label_tactic_indexed\' column\ndistinct_classes = train_data_assembled.select("label_tactic_indexed").distinct().collect()\n\n# Convert the result to a list of unique classes\nunique_classes = [row[\'label_tactic_indexed\'] for row in distinct_classes]\n\n# Print the unique classes\nprint("Unique classes:", unique_classes)\n\n# Group by the \'label_tactic_indexed\' column and count the occurrences of each class\nclass_counts = train_data_assembled.groupBy("label_tactic_indexed").count()\n\n# Show the class counts\nclass_counts.show()\n\n# Get the distinct values of the \'label_tactic_indexed\' column\ndistinct_classes = test_data_assembled.select("label_tactic_indexed").distinct().collect()\n\n# Convert the result to a list of unique classes\nunique_classes = [row[\'label_tactic_indexed\'] for row in distinct_classes]\n\n# Print the unique classes\nprint("Unique classes:", unique_classes)\n\n# Group by the \'label_tactic_indexed\' column and count 

In [12]:
from pyspark.ml.feature import StandardScaler

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic_indexed")

24/04/07 23:28:42 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:30:13 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB


In [13]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic_indexed")

In [14]:
# Define the PCA model
pca = PCA(k=8, inputCol="features_normalized", outputCol="pca_features")

# Fit the PCA model on the normalized training set
start_time = time.time()
pca_model = pca.fit(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/07 23:30:47 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:31:28 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:32:02 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:33:27 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:33:53 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:34:26 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:34:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/04/07 23:35:46 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


Execution time: 333.0059161186218 seconds


24/04/07 23:35:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [15]:
# Apply PCA transformation to the training and test sets
train_pca = pca_model.transform(train_data_normalized)
test_pca = pca_model.transform(test_data_normalized)

In [16]:
# Drop the normalized column and rename the pca_features column
train_pca = train_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
test_pca = test_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")

# Verify the changes
train_pca.show()
test_pca.show()

24/04/07 23:36:05 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


+--------------------+--------------------+
|label_tactic_indexed|            features|
+--------------------+--------------------+
|                 0.0|[0.00381634572699...|
|                 0.0|[0.00364108990037...|
|                 0.0|[0.00290472820931...|
|                 0.0|[-0.0030413202161...|
|                 0.0|[3.85918103803854...|
|                 0.0|[3.85919609842803...|
|                 0.0|[3.86197603010916...|
|                 0.0|[3.86228771616004...|
|                 0.0|[3.86228771616004...|
|                 0.0|[3.86078110794005...|
|                 0.0|[3.86078602694813...|
|                 0.0|[3.86078602694813...|
|                 0.0|[3.86356380420025...|
|                 0.0|[3.86515157882944...|
|                 0.0|[3.85916843321523...|
|                 0.0|[3.85916843321523...|
|                 0.0|[3.85918533502756...|
|                 0.0|[3.86037296361757...|
|                 0.0|[3.86075066270614...|
|                 0.0|[3.8607534

24/04/07 23:36:38 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


+--------------------+--------------------+
|label_tactic_indexed|            features|
+--------------------+--------------------+
|                 0.0|[0.00424598317500...|
|                 0.0|[3.85918103803854...|
|                 0.0|[3.85920069690697...|
|                 0.0|[3.85920069690697...|
|                 0.0|[3.86076604746783...|
|                 0.0|[3.86076604746783...|
|                 0.0|[3.86078110794005...|
|                 0.0|[3.86356380420025...|
|                 0.0|[3.86515157882944...|
|                 0.0|[3.85916565323073...|
|                 0.0|[3.85916565323073...|
|                 0.0|[3.85918533502756...|
|                 0.0|[3.85918655899770...|
|                 0.0|[3.85918655899770...|
|                 0.0|[3.86037296361757...|
|                 0.0|[3.86075066270614...|
|                 0.0|[3.86075344258596...|
|                 0.0|[3.86077156850313...|
|                 0.0|[3.86255864737474...|
|                 0.0|[3.8604706

In [17]:
# Create the SVM model
#svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=1)
#svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", predictionCol='prediction', maxIter=100, regParam=0.0, tol=.000001, fitIntercept=True, 
#                standardization=True, threshold=0.0, weightCol=None, aggregationDepth=2, maxBlockSizeInMB=0.0)
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)

# GOOD: labelCol, featuresCol, maxIter=1
# GOOD (probably):  regParam=0.0, tol=.000001, fitIntercept=True, standardization=True,
# BAD: threshold=0.0 (java heap out of memory)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/07 23:37:10 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB
24/04/07 23:38:25 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:39:55 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:40:27 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:40:45 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:41:19 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:41:39 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:42:00 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:42:23 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:42:44 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:43:07 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/07 23:43:29 WARN DAGSchedu

Execution time: 6495.150075674057 seconds


In [18]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.4170658588409424 seconds


In [19]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)

24/04/08 01:25:53 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/08 01:28:52 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/08 01:31:49 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/08 01:34:46 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


Accuracy: 0.9998904594489859
Precision: 0.999781122483
Recall: 0.9998904594489859
F1-Score: 0.9998357849640462


In [20]:
spark.sparkContext.stop()